# 01 - Exploracao STJ Metadados

Objetivo: explorar apenas os metadados da base de integras do STJ, sem carregar os textos grandes do ZIP.

Esta etapa descreve volume, periodo, tipos de documento, ministros, teor, recurso, assuntos codificados e contagens ano a ano.


## 1. Ambiente

No Colab, clone o repositorio antes de abrir/rodar o notebook, ou abra o notebook diretamente pelo GitHub. Os dados brutos devem ficar no Google Drive, fora do Git.

In [ ]:
# Rode esta celula no Colab se as dependencias ainda nao estiverem instaladas.
# !pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import re
import zipfile

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT_POINT = Path("/content/drive")

if (MOUNT_POINT / "MyDrive").exists():
    print("Google Drive já está montado.")
else:
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_DATA = Path("/content/drive/MyDrive/Mestrado/2026/llms/data")
RAW_STJ = DRIVE_DATA / "raw" / "stj_integras"
INTERIM_DATA = DRIVE_DATA / "interim"
PROCESSED_DATA = DRIVE_DATA / "processed"
REPORTS_DATA = DRIVE_DATA / "reports" / "summaries"

for path in [RAW_STJ, INTERIM_DATA, PROCESSED_DATA, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print("RAW_STJ:", RAW_STJ)
print("Arquivos encontrados:")
for file in sorted(RAW_STJ.iterdir()):
    print("-", file.name)


## 2. Inspecao dos arquivos brutos

Esperado em `data/raw/stj_integras/` no Drive:

- um JSON de metadados;
- um ZIP com textos integrais em TXT;
- um CSV com dicionario de dados.

In [ ]:
def inspect_raw_folder(raw_dir: str | Path) -> dict[str, list[Path]]:
    raw_dir = Path(raw_dir)
    files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
    return {
        'json': [p for p in files if p.suffix.lower() == '.json'],
        'zip': [p for p in files if p.suffix.lower() == '.zip'],
        'csv': [p for p in files if p.suffix.lower() == '.csv'],
        'other': [p for p in files if p.suffix.lower() not in {'.json', '.zip', '.csv'}],
    }

raw_files = inspect_raw_folder(RAW_STJ)
for kind, files in raw_files.items():
    print(kind.upper())
    for file in files:
        print(' ', file.name)

In [ ]:
# Ajuste manualmente caso a pasta tenha mais de um arquivo do mesmo tipo.
METADATA_PATH = raw_files["json"][0] if raw_files["json"] else None
ZIP_PATH = raw_files["zip"][0] if raw_files["zip"] else None
DICTIONARY_PATH = raw_files["csv"][0] if raw_files["csv"] else None

if METADATA_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .json encontrado em {RAW_STJ}")
if ZIP_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .zip encontrado em {RAW_STJ}")

print("METADATA_PATH:", METADATA_PATH)
print("ZIP_PATH:", ZIP_PATH)
print("DICTIONARY_PATH:", DICTIONARY_PATH)


## 3. Leitura dos metadados

In [ ]:
def load_metadata(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        if len(list_values) == 1:
            df = pd.DataFrame(list_values[0])
        else:
            df = pd.json_normalize(data)
    else:
        raise ValueError(f'Formato JSON inesperado: {type(data)}')

    print(f'Metadados: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
    return df

metadata_df = load_metadata(METADATA_PATH)
metadata_df.head()

In [ ]:
metadata_df.info()

In [ ]:
metadata_df.columns.tolist()

## 4. Analise exploratoria dos metadados

Antes de analisar texto integral, esta etapa descreve o universo de documentos e processos disponivel nos metadados: volume por ano, tipo de documento, ministro, teor, recurso e assuntos.

In [ ]:
def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized = {col.casefold(): col for col in df.columns}
    for candidate in candidates:
        if candidate.casefold() in normalized:
            return normalized[candidate.casefold()]
    return None

DATE_COLUMNS = {
    "dataPublicacao": ["dataPublicacao"],
    "dataRecebimento": ["dataRecebimento"],
    "dataDistribuicao": ["dataDistribuicao", "dataDistribuição"],
}

MINISTER_COLUMN = find_column(metadata_df, ["ministro", "NM_MINISTRO"])
PROCESS_COLUMN = find_column(metadata_df, ["processo"])
REGISTRATION_COLUMN = find_column(metadata_df, ["numeroRegistro"])

metadata_eda = metadata_df.copy()

for canonical_name, candidates in DATE_COLUMNS.items():
    source_col = find_column(metadata_eda, candidates)
    if source_col:
        metadata_eda[f"{canonical_name}_dt"] = pd.to_datetime(
            metadata_eda[source_col],
            errors="coerce",
            dayfirst=True,
        )
        metadata_eda[f"ano_{canonical_name}"] = metadata_eda[f"{canonical_name}_dt"].dt.year

if MINISTER_COLUMN and "ministro" not in metadata_eda.columns:
    metadata_eda["ministro"] = metadata_eda[MINISTER_COLUMN]

print("Coluna de ministro:", MINISTER_COLUMN)
print("Coluna de processo:", PROCESS_COLUMN)
print("Coluna de registro:", REGISTRATION_COLUMN)
metadata_eda.head()

In [ ]:
def count_docs_by(df: pd.DataFrame, column: str, n: int | None = None) -> pd.DataFrame:
    if column not in df.columns:
        return pd.DataFrame(columns=[column, "documentos"])
    counts = (
        df[column]
        .fillna("(vazio)")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="documentos")
    )
    return counts.head(n) if n else counts

summary = {
    "documentos": len(metadata_eda),
    "processos_unicos": metadata_eda[PROCESS_COLUMN].nunique() if PROCESS_COLUMN else None,
    "registros_unicos": metadata_eda[REGISTRATION_COLUMN].nunique() if REGISTRATION_COLUMN else None,
    "seqdocumento_unicos": metadata_eda["SeqDocumento"].nunique() if "SeqDocumento" in metadata_eda.columns else None,
    "data_publicacao_min": metadata_eda["dataPublicacao_dt"].min() if "dataPublicacao_dt" in metadata_eda.columns else None,
    "data_publicacao_max": metadata_eda["dataPublicacao_dt"].max() if "dataPublicacao_dt" in metadata_eda.columns else None,
}

summary

In [ ]:
docs_by_publication_year = count_docs_by(metadata_eda, "ano_dataPublicacao")
docs_by_publication_year

In [ ]:
if PROCESS_COLUMN and "ano_dataPublicacao" in metadata_eda.columns:
    processes_by_publication_year = (
        metadata_eda
        .dropna(subset=["ano_dataPublicacao"])
        .groupby("ano_dataPublicacao")[PROCESS_COLUMN]
        .nunique()
        .reset_index(name="processos_unicos")
        .sort_values("ano_dataPublicacao")
    )
else:
    processes_by_publication_year = pd.DataFrame(columns=["ano_dataPublicacao", "processos_unicos"])

processes_by_publication_year

In [ ]:
docs_by_type = count_docs_by(metadata_eda, "tipoDocumento")
docs_by_type.head(30)

In [ ]:
if "ano_dataPublicacao" in metadata_eda.columns and "tipoDocumento" in metadata_eda.columns:
    docs_by_year_type = pd.crosstab(
        metadata_eda["ano_dataPublicacao"],
        metadata_eda["tipoDocumento"],
        margins=True,
    )
else:
    docs_by_year_type = pd.DataFrame()

docs_by_year_type

In [ ]:
docs_by_minister = count_docs_by(metadata_eda, "ministro")
docs_by_minister.head(30)

In [ ]:
docs_by_teor = count_docs_by(metadata_eda, "teor")
docs_by_teor.head(30)

In [ ]:
docs_by_recurso = count_docs_by(metadata_eda, "recurso")
docs_by_recurso.head(30)

In [ ]:
TOP_N_ASSUNTOS = 20

# O campo "assuntos" vem como codigo/trilha hierarquica, nao como rotulo textual.
# Exemplo: "00899.07681.09580.09607.". Nesta etapa contamos os codigos
# sem tentar inferir nomes; o mapeamento para descricoes deve vir de uma tabela
# propria de assuntos/classes, se ela for incorporada depois.
def split_assunto_paths(value) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"\s*[;|,]\s*", text)
    return [part.strip() for part in parts if part.strip()]


def assunto_levels(path: str) -> dict[str, str | None]:
    codes = [part for part in str(path).split(".") if part]
    return {
        "assunto_raiz": codes[0] if codes else None,
        "assunto_final": codes[-1] if codes else None,
        "assunto_profundidade": len(codes),
    }

if "assuntos" in metadata_eda.columns:
    assuntos_long = (
        metadata_eda[["SeqDocumento", "ano_dataPublicacao", "assuntos"]]
        .assign(assunto_path=lambda df: df["assuntos"].map(split_assunto_paths))
        .explode("assunto_path")
        .dropna(subset=["assunto_path"])
    )
    assunto_levels_df = pd.DataFrame(
        assuntos_long["assunto_path"].map(assunto_levels).tolist(),
        index=assuntos_long.index,
    )
    assuntos_long = pd.concat(
        [assuntos_long.drop(columns=["assuntos"]), assunto_levels_df],
        axis=1,
    )
else:
    assuntos_long = pd.DataFrame(
        columns=["SeqDocumento", "ano_dataPublicacao", "assunto_path", "assunto_raiz", "assunto_final", "assunto_profundidade"]
    )

docs_by_assunto_path = (
    assuntos_long["assunto_path"]
    .value_counts()
    .rename_axis("assunto_path")
    .reset_index(name="ocorrencias")
)
docs_by_assunto_root = (
    assuntos_long["assunto_raiz"]
    .value_counts()
    .rename_axis("assunto_raiz")
    .reset_index(name="ocorrencias")
)
docs_by_assunto_final = (
    assuntos_long["assunto_final"]
    .value_counts()
    .rename_axis("assunto_final")
    .reset_index(name="ocorrencias")
)

top_assunto_paths = docs_by_assunto_path.head(TOP_N_ASSUNTOS)["assunto_path"].tolist()
top_assunto_finais = docs_by_assunto_final.head(TOP_N_ASSUNTOS)["assunto_final"].tolist()

docs_by_year_top_assunto_path = pd.crosstab(
    assuntos_long.loc[assuntos_long["assunto_path"].isin(top_assunto_paths), "ano_dataPublicacao"],
    assuntos_long.loc[assuntos_long["assunto_path"].isin(top_assunto_paths), "assunto_path"],
)

docs_by_year_top_assunto_final = pd.crosstab(
    assuntos_long.loc[assuntos_long["assunto_final"].isin(top_assunto_finais), "ano_dataPublicacao"],
    assuntos_long.loc[assuntos_long["assunto_final"].isin(top_assunto_finais), "assunto_final"],
)

docs_by_assunto_final.head(30)


In [ ]:
metadata_reports = {
    "stj_docs_by_publication_year.csv": docs_by_publication_year,
    "stj_processes_by_publication_year.csv": processes_by_publication_year,
    "stj_docs_by_type.csv": docs_by_type,
    "stj_docs_by_minister.csv": docs_by_minister,
    "stj_docs_by_teor.csv": docs_by_teor,
    "stj_docs_by_recurso.csv": docs_by_recurso,
    "stj_docs_by_assunto_path.csv": docs_by_assunto_path,
    "stj_docs_by_assunto_root.csv": docs_by_assunto_root,
    "stj_docs_by_assunto_final.csv": docs_by_assunto_final,
    "stj_docs_by_year_top_assunto_path.csv": docs_by_year_top_assunto_path.reset_index(),
    "stj_docs_by_year_top_assunto_final.csv": docs_by_year_top_assunto_final.reset_index(),
}

if not docs_by_year_type.empty:
    metadata_reports["stj_docs_by_year_type.csv"] = docs_by_year_type.reset_index()

for filename, table in metadata_reports.items():
    output_path = REPORTS_DATA / filename
    table.to_csv(output_path, index=False)
    print("Salvo:", output_path)


In [ ]:
metadata_summary_md = f"""# EDA de metadados - STJ Integras

## Totais

- Documentos: {summary['documentos']:,}
- Processos unicos: {summary['processos_unicos']:,}
- Registros unicos: {summary['registros_unicos']:,}
- SeqDocumento unicos: {summary['seqdocumento_unicos']:,}
- Periodo de publicacao: {summary['data_publicacao_min']} a {summary['data_publicacao_max']}

## Documentos por ano de publicacao

{docs_by_publication_year.to_markdown(index=False)}

## Processos unicos por ano de publicacao

{processes_by_publication_year.to_markdown(index=False)}

## Tipos de documento mais frequentes

{docs_by_type.head(20).to_markdown(index=False)}

## Ministros mais frequentes

{docs_by_minister.head(20).to_markdown(index=False)}

## Teor

{docs_by_teor.head(20).to_markdown(index=False)}

## Recursos

{docs_by_recurso.head(20).to_markdown(index=False)}

## Assuntos

O campo `assuntos` foi tratado como codigo/trilha hierarquica. Os codigos abaixo ainda nao foram mapeados para rotulos textuais.

### Codigos finais de assunto mais frequentes

{docs_by_assunto_final.head(20).to_markdown(index=False)}

### Trilhas completas de assunto mais frequentes

{docs_by_assunto_path.head(20).to_markdown(index=False)}

### Codigos raiz de assunto mais frequentes

{docs_by_assunto_root.head(20).to_markdown(index=False)}

## Observacao sobre cruzamentos ano a ano

Contagens por ano para documentos e processos sao pequenas e legiveis. Para assuntos, o cruzamento completo pode ficar muito esparso; por isso o notebook salva apenas os Top {TOP_N_ASSUNTOS} assuntos por caminho completo e por codigo final.
"""

metadata_summary_path = REPORTS_DATA / "metadata_eda_summary.md"
metadata_summary_path.write_text(metadata_summary_md, encoding="utf-8")
metadata_summary_path


## 5. Proximos passos

- Revisar os CSVs e `metadata_eda_summary.md` em `reports/summaries`.
- Se o recorte fizer sentido, rodar `02_validacao_integras_txt.ipynb` para validar a ligacao com os arquivos TXT.
- Manter a analise textual para uma etapa posterior, com amostra controlada.
